In [4]:
import ee
import geemap
import datetime
import osmnx as ox
import geopandas as gpd
import shapely
import shapely.geometry as geom
from shapely.geometry import Polygon


In [3]:
!pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 6.2 MB/s eta 0:00:00


In [5]:
ee.Authenticate()

In [6]:
ee.Initialize(project = 'ee-kubek114')

In [10]:
# 1. PARAMETRY UŻYTKOWNIKA
m = geemap.Map()
m.add_basemap('Esri.WorldImagery')

# Lokalizacja: środek obszaru + promień (km)

center_lat = 50.4700
center_lon = 17.3340
radius_km = 1

# Data zdarzenia
event_date_str = '2024-09-13'
days_before = 12
days_after = 6

# 2. GEOMETRIA OBSZARU

center_point = ee.Geometry.Point([center_lon, center_lat])
#area = center_point.buffer(radius_km * 1000)
#m.add_layer(area, {'color': 'red'}, 'Obszar analizy')

# Opcjonalnie "area" jako gmina Nysa
nysa_gmina = ee.FeatureCollection("projects/ee-kubek114/assets/Nysa_gmina")
area = nysa_gmina.geometry()

area_outline = ee.FeatureCollection(area).style(
    color='red',
    width=2,
    fillColor='00000000'
)

m.add_layer(area_outline, {}, 'Granica Nysy')
m.centerObject(area, 12)
m

Map(center=[50.45758765104263, 17.353593719899337], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
# Import sieci drogowej z OSM
nysa_gmina = ee.FeatureCollection("projects/ee-kubek114/assets/Nysa_gmina")
area = nysa_gmina.geometry()

area_gdf = geemap.ee_to_gdf(ee.FeatureCollection([ee.Feature(area)]))
area_gdf = area_gdf.set_crs(epsg=4326)

minx, miny, maxx, maxy = area_gdf.total_bounds
bbox = (minx, miny, maxx, maxy)
G = ox.graph_from_bbox(
    bbox=bbox,
    network_type="drive_service"
)

edges = ox.graph_to_gdfs(G, nodes=False, edges=True)
edges_wgs = edges.to_crs(epsg=4326)

In [ ]:
%%time
# 3. AUTOMATYCZNE WYZNACZENIE OKIEN CZASOWYCH

event_date = datetime.date.fromisoformat(event_date_str)

before_start = (event_date - datetime.timedelta(days=days_before)).isoformat()
before_end   = (event_date - datetime.timedelta(days=1)).isoformat()

after_start  = (event_date + datetime.timedelta(days=1)).isoformat()
after_end    = (event_date + datetime.timedelta(days=days_after)).isoformat()


# 2. GEOMETRIA OBSZARU

center_point = ee.Geometry.Point([center_lon, center_lat])

#area = center_point.buffer(radius_km * 1000)

#m.add_layer(area, {'color': 'red'}, 'Obszar analizy')

# Opcjonalnie "area" jako gmina Nysa
nysa_gmina = ee.FeatureCollection("projects/ee-kubek114/assets/Nysa_gmina")
area = nysa_gmina.geometry()

nysa_outline = nysa_gmina.style(
    color='red',
    width=2,
    fillColor='00000000'
)
m.add_layer(nysa_outline, {}, 'Granica Nysy')
m.centerObject(area, 12)

# 4. PROSTY TRYB WYBORU ORBIT I PARAMETRÓW

INSTRUMENT_MODE = 'IW'
POLARIZATIONS   = ['VV', 'VH']

USE_DESCENDING = True
USE_ASCENDING  = True

# 5. BUDOWANIE KOLEKCJI SENTINEL-1

collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filter(ee.Filter.eq('instrumentMode', INSTRUMENT_MODE))
    .filterBounds(area)
    .select(POLARIZATIONS)
)

# Before

before_coll = collection.filterDate(before_start, before_end)

if USE_DESCENDING:
    before_desc_coll = before_coll.filter(
        ee.Filter.eq('orbitProperties_pass', 'DESCENDING')
    )
else:
    before_desc_coll = ee.ImageCollection([])

if USE_ASCENDING:
    before_asc_coll = before_coll.filter(
        ee.Filter.eq('orbitProperties_pass', 'ASCENDING')
    )
else:
    before_asc_coll = ee.ImageCollection([])

# After

after_coll = collection.filterDate(after_start, after_end)

if USE_DESCENDING:
    after_desc_coll = after_coll.filter(
        ee.Filter.eq('orbitProperties_pass', 'DESCENDING')
    )
else:
    after_desc_coll = ee.ImageCollection([])

if USE_ASCENDING:
    after_asc_coll = after_coll.filter(
        ee.Filter.eq('orbitProperties_pass', 'ASCENDING')
    )
else:
    after_asc_coll = ee.ImageCollection([])

# Moizaiki zbiorów

before_desc = before_desc_coll.mosaic().clip(area)
before_asc = before_asc_coll.mosaic().clip(area)

after_desc = after_desc_coll.mosaic().clip(area)
after_asc = after_asc_coll.mosaic().clip(area)

# Definicje filtru Refined Lee

def to_natural(img):
    """Konwersja z dB do skali liniowej."""
    return ee.Image(10.0).pow(img.divide(10.0))

def to_db(img):
    """Konwersja ze skali liniowej do dB."""
    return img.log10().multiply(10.0)

def refinedLee_function(img):

    weights3 = ee.List.repeat(ee.List.repeat(1, 3), 3)
    kernel3 = ee.Kernel.fixed(3, 3, weights3, 1, 1, False)

    mean3 = img.reduceNeighborhood(ee.Reducer.mean(), kernel3)
    var3 = img.reduceNeighborhood(ee.Reducer.variance(), kernel3)

    sample_weights = ee.List([
        [0,0,0,0,0,0,0],
        [0,1,0,1,0,1,0],
        [0,0,0,0,0,0,0],
        [0,1,0,1,0,1,0],
        [0,0,0,0,0,0,0],
        [0,1,0,1,0,1,0],
        [0,0,0,0,0,0,0]
    ])
    sample_kernel = ee.Kernel.fixed(7, 7, sample_weights, 3, 3, False)

    sample_mean = mean3.neighborhoodToBands(sample_kernel)
    sample_var = var3.neighborhoodToBands(sample_kernel)

    gradients = sample_mean.select(1).subtract(sample_mean.select(7)).abs() \
        .addBands(sample_mean.select(6).subtract(sample_mean.select(2)).abs()) \
        .addBands(sample_mean.select(3).subtract(sample_mean.select(5)).abs()) \
        .addBands(sample_mean.select(0).subtract(sample_mean.select(8)).abs())

    max_gradient = gradients.reduce(ee.Reducer.max())
    gradmask = gradients.eq(max_gradient)
    gradmask = gradmask.addBands(gradmask)

    directions = sample_mean.select(1).subtract(sample_mean.select(4)) \
        .gt(sample_mean.select(4).subtract(sample_mean.select(7))).multiply(1) \
        .addBands(
            sample_mean.select(6).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(2))).multiply(2)
        ).addBands(
            sample_mean.select(3).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(5))).multiply(3)
        ).addBands(
            sample_mean.select(0).subtract(sample_mean.select(4)) \
            .gt(sample_mean.select(4).subtract(sample_mean.select(8))).multiply(4)
        )

    directions = directions.addBands(directions.select(0).Not().multiply(5)) \
        .addBands(directions.select(1).Not().multiply(6)) \
        .addBands(directions.select(2).Not().multiply(7)) \
        .addBands(directions.select(3).Not().multiply(8))

    directions = directions.updateMask(gradmask)
    directions = directions.reduce(ee.Reducer.sum())

    sample_stats = sample_var.divide(sample_mean.multiply(sample_mean))
    sigmaV = sample_stats.toArray().arraySort().arraySlice(0, 0, 5).arrayReduce(ee.Reducer.mean(), [0])

    rect_weights = ee.List.repeat(ee.List.repeat(0,7),3).cat(ee.List.repeat(ee.List.repeat(1,7),4))
    diag_weights = ee.List([
        [1,0,0,0,0,0,0],
        [1,1,0,0,0,0,0],
        [1,1,1,0,0,0,0],
        [1,1,1,1,0,0,0],
        [1,1,1,1,1,0,0],
        [1,1,1,1,1,1,0],
        [1,1,1,1,1,1,1]
    ])

    rect_kernel = ee.Kernel.fixed(7, 7, rect_weights, 3, 3, False)
    diag_kernel = ee.Kernel.fixed(7, 7, diag_weights, 3, 3, False)

    dir_mean = img.reduceNeighborhood(ee.Reducer.mean(), rect_kernel).updateMask(directions.eq(1))
    dir_var = img.reduceNeighborhood(ee.Reducer.variance(), rect_kernel).updateMask(directions.eq(1))

    dir_mean = dir_mean.addBands(img.reduceNeighborhood(ee.Reducer.mean(), diag_kernel).updateMask(directions.eq(2)))
    dir_var = dir_var.addBands(img.reduceNeighborhood(ee.Reducer.variance(), diag_kernel).updateMask(directions.eq(2)))

    for i in range(1, 4):
        dir_mean = dir_mean.addBands(
            img.reduceNeighborhood(ee.Reducer.mean(), rect_kernel.rotate(i)).updateMask(directions.eq(2*i+1))
        )
        dir_var = dir_var.addBands(
            img.reduceNeighborhood(ee.Reducer.variance(), rect_kernel.rotate(i)).updateMask(directions.eq(2*i+1))
        )
        dir_mean = dir_mean.addBands(
            img.reduceNeighborhood(ee.Reducer.mean(), diag_kernel.rotate(i)).updateMask(directions.eq(2*i+2))
        )
        dir_var = dir_var.addBands(
            img.reduceNeighborhood(ee.Reducer.variance(), diag_kernel.rotate(i)).updateMask(directions.eq(2*i+2))
        )

    dir_mean = dir_mean.reduce(ee.Reducer.sum())
    dir_var = dir_var.reduce(ee.Reducer.sum())

    varX = dir_var.subtract(dir_mean.multiply(dir_mean).multiply(sigmaV)).divide(sigmaV.add(1.0))
    b = varX.divide(dir_var)

    result = dir_mean.add(b.multiply(img.subtract(dir_mean)))

    return result.arrayFlatten([['sum']])


def refinedLee(image):
    """
    Zastosuj filtr Refined Lee do wszystkich pasm obrazu (np. VV, VH).
    Zakładamy, że wejściowy image jest w dB (tak jak Sentinel-1 GRD po kalibracji).
    Zwraca obraz w dB, z zachowanymi nazwami pasm i właściwościami.
    """
    props = image.toDictionary(image.propertyNames())
    bands = image.bandNames()

    def _filter_band(band_name):
        band_name = ee.String(band_name)
        band = image.select([band_name])
        # dB -> linear -> filtr -> dB
        filtered = to_db(refinedLee_function(to_natural(band)))
        return filtered.rename(band_name)

    filtered_ic = ee.ImageCollection(bands.map(_filter_band))
    filtered = filtered_ic.toBands().rename(bands)

    return filtered.set(props)

# Stotosanie filtrów

beforeD_filtered = refinedLee(before_desc)
beforeA_filtered = refinedLee(before_asc)
afterD_filtered  = refinedLee(after_desc)
afterA_filtered  = refinedLee(after_asc)

print("Filtry OK")

# Progowe wykrycie powodzi

beforeD_vh_lin = to_natural(beforeD_filtered.select('VH'))
afterD_vh_lin  = to_natural(afterD_filtered.select('VH'))

beforeA_vh_lin = to_natural(beforeA_filtered.select('VH'))
afterA_vh_lin  = to_natural(afterA_filtered.select('VH'))


diffD = beforeD_vh_lin.divide(afterD_vh_lin).rename('ratio')
diffA = beforeA_vh_lin.divide(afterA_vh_lin).rename('ratio')


flood_value = 1.35
floodedD = diffD.gt(flood_value).rename('water').selfMask()
floodedA = diffA.gt(flood_value).rename('water').selfMask()

# Filtorwanie obszarów permanentej wody

water_surface = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").clip(area)
seasonality = water_surface.select('seasonality')


permament_water = seasonality.gte(6)
permament_water_bin = permament_water.unmask(0)
non_permament_mask = permament_water_bin.Not()


# Filtorwanie obszarów o sporym nachyleniu

hydroshed = ee.Image('WWF/HydroSHEDS/03VFDEM').clip(area)
terrain = ee.Algorithms.Terrain(hydroshed)
slope = terrain.select('slope')

max_slope = 5
low_slope_mask = slope.lt(max_slope)

# Połączona maska

combined_mask = non_permament_mask.And(low_slope_mask)

floodedD_masked = floodedD.updateMask(combined_mask)
floodedA_masked = floodedA.updateMask(combined_mask)

# Operacje morfologiczne

baseD = floodedD_masked.unmask(0)
#floodedD_m = baseD.focal_max(radius=20, units='meters').focal_min(radius=20, units='meters') # zamkniecie
#floodedD_m = baseD.focal_min(radius=20, units='meters').focal_max(radius=20, units='meters') # otwarcie
floodedD_m = baseD.focal_max(radius=10, units='meters').focal_min(radius=15, units='meters').focal_max(radius=5, units='meters')
floodedD_morf = floodedD_m.updateMask(floodedD_m)

baseA = floodedA_masked.unmask(0)
floodedA_m = baseA.focal_max(radius=10, units='meters').focal_min(radius=15, units='meters').focal_max(radius=5, units='meters')
floodedA_morf = floodedA_m.updateMask(floodedA_m)

# Zmiana rozdzielczości

floodedD_10m = floodedD_morf.setDefaultProjection('EPSG:4326', None, 10) \
    .reduceResolution(reducer=ee.Reducer.max(), maxPixels=1024) \
    .reproject(crs='EPSG:4326', scale=10)
floodedA_10m = floodedA_morf.setDefaultProjection('EPSG:4326', None, 10) \
    .reduceResolution(reducer=ee.Reducer.max(), maxPixels=1024) \
    .reproject(crs='EPSG:4326', scale=10)

print("Operacje OK")

# wektoryzacja warst

flood_vectorsD = floodedD_10m.reduceToVectors(
    geometry=area,
    scale=10,
    crs='EPSG:4326',
    geometryType='polygon',
    eightConnected=True,
    labelProperty='water',
    maxPixels=1e10
)

flood_vectorsA = floodedA_10m.reduceToVectors(
    geometry=area,
    scale=10,
    crs='EPSG:4326',
    geometryType='polygon',
    eightConnected=True,
    labelProperty='water',
    maxPixels=1e10
)

# filtracja małych obszarów
min_area_m2 = 800

flood_vectorsD_area = flood_vectorsD.map(lambda f: f.set('area_m2', f.geometry().area(1)))
flood_vectors_filteredD = flood_vectorsD_area.filter(ee.Filter.gte('area_m2', min_area_m2))

print("Wektoryzacja OK")


# Tworzenie warstw wynikowych
from shapely.geometry import Point, MultiPoint, GeometryCollection

flood_gdfD = geemap.ee_to_gdf(flood_vectors_filteredD)

CRS_PROJ = "EPSG:3857"
roads_proj = edges.to_crs(CRS_PROJ)
flood_proj = flood_gdfD.to_crs(CRS_PROJ)

boundaries = flood_proj.copy()
boundaries["geometry"] = boundaries.geometry.boundary

boundary_union = boundaries.unary_union

# Punkty przecięcia
points_list = []
for ridx, row in roads_proj.iterrows():
    inter = row.geometry.intersection(boundary_union)
    if inter.is_empty:
        continue
    if isinstance(inter, Point):
        points_list.append({"geometry": inter, "road_id": ridx})
    elif isinstance(inter, MultiPoint):
        for pt in inter.geoms:
            points_list.append({"geometry": pt, "road_id": ridx})
    elif isinstance(inter, GeometryCollection):
        for geom_part in inter.geoms:
            if isinstance(geom_part, Point):
                points_list.append({"geometry": geom_part, "road_id": ridx})

if len(points_list) == 0:
    print("Brak punktów przecięcia dróg z granicą powodzi.")

else:
    points_gdf = gpd.GeoDataFrame(points_list, geometry="geometry", crs=CRS_PROJ)
    print(points_gdf.head())

    # Bufory 5 m wokół punktów
    buffers_gdf = points_gdf.copy()
    buffers_gdf["geometry"] = buffers_gdf.buffer(5)

# Drogi zalane
roads_in_flood = gpd.overlay(roads_proj, flood_proj, how="intersection")

# Drogi niezalane
roads_outside_flood = gpd.overlay(roads_proj, flood_proj, how="difference")

# Obliczanie statystyk obszarów zalania

# Obliczanie statystyk dróg

roads_in_flood["length_m"] = roads_in_flood.geometry.length
roads_outside_flood["length_m"] = roads_outside_flood.geometry.length

flooded_length_m = roads_in_flood["length_m"].sum()
dry_length_m = roads_outside_flood["length_m"].sum()

print("Zalane drogi:", flooded_length_m / 1000, "km")
print("Niezalane drogi:", dry_length_m / 1000, "km")


Filtry OK
Operacje OK
Wektoryzacja OK
                      geometry                     road_id
0  POINT (1930350 6530664.297)  (316584614, 2440292643, 0)
1  POINT (1930010 6530501.132)  (316584615, 2421091913, 0)
2   POINT (1927520 6530001.63)  (316584641, 2311178075, 0)
3  POINT (1927510 6530010.715)  (316584641, 2311178075, 0)
4  POINT (1930130 6529502.586)  (441255489, 2261800859, 0)
Zalane drogi: 114.99135536172011 km
Niezalane drogi: 2036.2255257300233 km
CPU times: user 13.5 s, sys: 49.1 ms, total: 13.5 s
Wall time: 23.3 s


In [ ]:
# Eksport warstw wynikowych

points_gdf.to_file("/content/intersection_points.shp")
buffers_gdf.to_file("/content/buffers_5m.shp")
roads_in_flood.to_file("/content/drogi_zalane.shp")
roads_outside_flood.to_file("/content/drogi_niezalane.shp")

flood_gdf_filteredD = geemap.ee_to_gdf(flood_vectors_filteredD)
flood_gdf_filteredD = flood_gdf_filteredD.set_crs(epsg=4326)
flood_gdf_filteredD_3857 = flood_gdf_filteredD.to_crs(epsg=3857)
flood_gdf_filteredD_3857.to_file("/content/zalane_sar.shp")

mask_img = permament_water_bin.rename("mask").selfMask()
mask_vectors = mask_img.reduceToVectors(
    geometry=area,
    scale=30,                 # JRC GSW jest 30 m
    crs="EPSG:4326",
    geometryType="polygon",
    eightConnected=True,
    labelProperty="mask",
    maxPixels=1e10
)
mask_gdf = geemap.ee_to_gdf(mask_vectors)  # albo mask_vectors
mask_gdf = mask_gdf.set_crs(epsg=4326)
mask_gdf_3857 = mask_gdf.to_crs(epsg=3857)
mask_gdf_3857.to_file("/content/permanent_water.shp")